In [8]:
import os, argparse, random, numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from _utils import MMDataset, overall_performance_report
from torch.distributions import NegativeBinomial
from pyro.distributions.zero_inflated import ZeroInflatedNegativeBinomial

class AE(nn.Module):
    def __init__(self, embed_dim=200, num_views=3, feature_dims=[1000, 1000, 500], hidden_dims=[512, 512, 512]):
        super(AE, self).__init__()
        self.embed_dim = embed_dim; self.num_views = num_views; self.feature_dims = feature_dims; self.hidden_dims = hidden_dims
        self.encoder_list = nn.ModuleList([nn.Sequential(nn.Linear(feature_dims[i], hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(),
                                                         nn.Linear(hidden_dims[i], embed_dim)) for i in range(num_views)])
        self.fusion_net = nn.Linear(num_views*embed_dim, embed_dim)
        self.decoder_list = nn.ModuleList([nn.Sequential(nn.Linear(embed_dim, hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(),
                                                         nn.Linear(hidden_dims[i], feature_dims[i])) for i in range(num_views)])
    
    def forward(self, x):
        encoded_output_list = [self.encoder_list[i](x[i]) for i in range(self.num_views)]
        encoded_output = self.fusion_net(torch.cat(encoded_output_list, dim=1))
        decoded_output_list = [self.decoder_list[i](encoded_output) for i in range(self.num_views)]
        return decoded_output_list, encoded_output
    
    def forward_embedding(self, x):
        encoded_output_list = [self.encoder_list[i](x[i]) for i in range(self.num_views)]
        encoded_output = self.fusion_net(torch.cat(encoded_output_list, dim=1))
        return encoded_output
    
    def reconstruction_loss(self, x):
        decoded_output_list, _ = self.forward(x)
        return sum([F.mse_loss(decoded_output_list[v], x[v], reduction='mean') for v in range(self.num_views)])
    
class DAE(nn.Module):
    def __init__(self, embed_dim=200, num_views=3, feature_dims=[1000, 1000, 500], hidden_dims=[512, 512, 512], noise_factor=0.1):
        super(DAE, self).__init__()
        self.embed_dim = embed_dim; self.num_views = num_views; self.feature_dims = feature_dims; self.hidden_dims = hidden_dims; self.noise_factor = noise_factor
        self.encoder_list = nn.ModuleList([nn.Sequential(nn.Linear(feature_dims[i], hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(),
                                                         nn.Linear(hidden_dims[i], embed_dim)) for i in range(num_views)])
        self.fusion_net = nn.Linear(num_views*embed_dim, embed_dim)
        self.decoder_list = nn.ModuleList([nn.Sequential(nn.Linear(embed_dim, hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(),
                                                         nn.Linear(hidden_dims[i], feature_dims[i])) for i in range(num_views)])
    
    def forward(self, x):
        if self.training:
            x = [x[i] + self.noise_factor * torch.randn_like(x[i]) for i in range(self.num_views)]
        encoded_output_list = [self.encoder_list[i](x[i]) for i in range(self.num_views)]
        encoded_output = self.fusion_net(torch.cat(encoded_output_list, dim=1))
        decoded_output_list = [self.decoder_list[i](encoded_output) for i in range(self.num_views)]
        return decoded_output_list, encoded_output
    
    def forward_embedding(self, x):
        encoded_output_list = [self.encoder_list[i](x[i]) for i in range(self.num_views)]
        encoded_output = self.fusion_net(torch.cat(encoded_output_list, dim=1))
        return encoded_output

    def reconstruction_loss(self, x):
        decoded_output_list, _ = self.forward(x)
        return sum([F.mse_loss(decoded_output_list[v], x[v], reduction='mean') for v in range(self.num_views)])
    
class VAE(nn.Module):
    def __init__(self, embed_dim=200, num_views=3, feature_dims=[1000, 1000, 500], hidden_dims=[512, 512, 512]):
        super(VAE, self).__init__()
        self.embed_dim = embed_dim; self.num_views = num_views; self.feature_dims = feature_dims; self.hidden_dims = hidden_dims
        self.encoder_list = nn.ModuleList([nn.Sequential(nn.Linear(feature_dims[i], hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(),
                                                         nn.Linear(hidden_dims[i], embed_dim)) for i in range(num_views)])
        self.fusion_net_mean = nn.Linear(num_views*embed_dim, embed_dim)
        self.fusion_net_logvar = nn.Linear(num_views*embed_dim, embed_dim)
        self.decoder_list = nn.ModuleList([nn.Sequential(nn.Linear(embed_dim, hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(),
                                                         nn.Linear(hidden_dims[i], feature_dims[i])) for i in range(num_views)])
    
    def forward(self, x):
        encoded_output_list = [self.encoder_list[i](x[i]) for i in range(self.num_views)]
        mean = self.fusion_net_mean(torch.cat(encoded_output_list, dim=1))
        logvar = self.fusion_net_logvar(torch.cat(encoded_output_list, dim=1))
        z = self.reparameterize(mean, logvar)
        decoded_output_list = [self.decoder_list[i](z) for i in range(self.num_views)]
        return decoded_output_list, mean, logvar
    
    def forward_embedding(self, x):
        encoded_output_list = [self.encoder_list[i](x[i]) for i in range(self.num_views)]
        mean = self.fusion_net_mean(torch.cat(encoded_output_list, dim=1))
        return mean
    
    def reparameterize(self, mean, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mean + eps * std
    
    def reconstruction_loss(self, x):
        decoded_output_list, mean, logvar = self.forward(x)
        reconstruction_loss = sum([F.mse_loss(decoded_output_list[v], x[v], reduction='mean') for v in range(self.num_views)])
        kld_loss = -0.5 * torch.sum(1 + logvar - mean.pow(2) - logvar.exp())
        return reconstruction_loss + kld_loss
    
class ZINBVAE(nn.Module):
    def __init__(self, embed_dim=200, num_views=3, feature_dims=[1000, 1000, 500], hidden_dims=[512, 512, 512], distribution='zinb'):
        super(ZINBVAE, self).__init__()
        self.embed_dim = embed_dim; self.num_views = num_views; self.feature_dims = feature_dims; self.hidden_dims = hidden_dims; self.distribution = distribution
        self.encoder_list = nn.ModuleList([nn.Sequential(nn.Linear(feature_dims[i], hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU(),
                                                         nn.Linear(hidden_dims[i], embed_dim)) for i in range(num_views)])
        self.fusion_net_mean = nn.Linear(num_views*embed_dim, embed_dim)
        self.fusion_net_logvar = nn.Linear(num_views*embed_dim, embed_dim)
        self.decoder_list = nn.ModuleList([nn.Sequential(nn.Linear(embed_dim, hidden_dims[i]), nn.BatchNorm1d(hidden_dims[i]), nn.ReLU()) for i in range(num_views)])
        self.zinb_log_mean_list = nn.ModuleList([nn.Linear(hidden_dims[i], feature_dims[i]) for i in range(num_views)])
        self.zinb_dropout_list = nn.ModuleList([nn.Linear(hidden_dims[i], feature_dims[i]) for i in range(num_views)])
        self.zinb_log_theta_list = nn.ParameterList([nn.Parameter(torch.randn(feature_dims[i])) for i in range(num_views)])
    
    def forward(self, x):
        encoded_output_list = [self.encoder_list[i](x[i]) for i in range(self.num_views)]
        mean = self.fusion_net_mean(torch.cat(encoded_output_list, dim=1))
        logvar = self.fusion_net_logvar(torch.cat(encoded_output_list, dim=1))
        z = self.reparameterize(mean, logvar)
        decoded_output_list = [self.decoder_list[i](z) for i in range(self.num_views)]
        zinb_log_mean_list = [self.zinb_log_mean_list[i](decoded_output_list[i]) for i in range(self.num_views)]
        zinb_dropout_list = [self.zinb_dropout_list[i](decoded_output_list[i]) for i in range(self.num_views)]
        zinb_log_theta_list = [self.zinb_log_theta_list[i] for i in range(self.num_views)]
        return decoded_output_list, mean, logvar, zinb_log_mean_list, zinb_dropout_list, zinb_log_theta_list
    
    def forward_embedding(self, x):
        encoded_output_list = [self.encoder_list[i](x[i]) for i in range(self.num_views)]
        mean = self.fusion_net_mean(torch.cat(encoded_output_list, dim=1))
        return mean
    
    def reparameterize(self, mean, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mean + eps * std
    
    def reconstruction_loss(self, x):
        decoded_output_list, mean, logvar = self.forward(x)
        reconstruction_loss = sum([F.mse_loss(decoded_output_list[v], x[v], reduction='mean') for v in range(self.num_views)])
        kld_loss = -0.5 * torch.sum(1 + logvar - mean.pow(2) - logvar.exp())
        return reconstruction_loss + kld_loss
    
    def generate(self, x, random=False):
        encoded_output_list = [self.encoder_list[i](x[i]) for i in range(self.num_views)]
        mean = self.fusion_net_mean(torch.cat(encoded_output_list, dim=1))
        logvar = self.fusion_net_logvar(torch.cat(encoded_output_list, dim=1))
        z = self.reparameterize(mean, logvar) if random else mean
        decoded_output_list = [self.decoder_list[i](z) for i in range(self.num_views)]
        zinb_log_mean_list = [self.zinb_log_mean_list[i](decoded_output_list[i]) for i in range(self.num_views)]
        zinb_dropout_list = [self.zinb_dropout_list[i](decoded_output_list[i]) for i in range(self.num_views)]
        zinb_log_theta_list = [self.zinb_log_theta_list[i] for i in range(self.num_views)]
        zinb_mean_list = [zinb_log_mean_list[i].exp() for i in range(self.num_views)] # element shape: [batch_size, feature_dim]
        zinb_theta_list = [zinb_log_theta_list[i].exp() for i in range(self.num_views)] # element shape: [batch_size, feature_dim]
        nb_logits_list = [((zinb_mean_list[i] + 1e-4).log() - (zinb_theta_list[i] + 1e-4).log()) for i in range(self.num_views)] # element shape: [batch_size, feature_dim]
        if self.distribution == 'zinb':
            distribution = [ZeroInflatedNegativeBinomial(total_count=zinb_theta_list[i], logits=nb_logits_list[i], gate_logits = zinb_dropout_list[i], validate_args=False) for i in range(self.num_views)]
            generated_output_list = [distribution[i].sample() if random else distribution[i].mean for i in range(self.num_views)]
        elif self.distribution == 'nb':
            distribution = [NegativeBinomial(total_count=zinb_theta_list[i], logits=nb_logits_list[i], validate_args=False) for i in range(self.num_views)]
            generated_output_list = [distribution[i].sample() if random else distribution[i].mean for i in range(self.num_views)]
        return generated_output_list # shape: [num_views, batch_size, feature_dim]
    
    def reconstruction_loss(self, x):
        decoded_output_list, mean, logvar, zinb_log_mean_list, zinb_dropout_list, zinb_log_theta_list = self.forward(x)
        zinb_mean_list = [zinb_log_mean_list[i].exp() for i in range(self.num_views)] # element shape: [batch_size, feature_dim]
        zinb_theta_list = [zinb_log_theta_list[i].exp() for i in range(self.num_views)] # element shape: [batch_size, feature_dim]
        nb_logits_list = [((zinb_mean_list[i] + 1e-4).log() - (zinb_theta_list[i] + 1e-4).log()) for i in range(self.num_views)] # element shape: [batch_size, feature_dim]
        if self.distribution == 'zinb':
            distribution = [ZeroInflatedNegativeBinomial(total_count=zinb_theta_list[i], logits=nb_logits_list[i], gate_logits = zinb_dropout_list[i], validate_args=False) for i in range(self.num_views)]
        elif self.distribution == 'nb':
            distribution = [NegativeBinomial(total_count=zinb_theta_list[i], logits=nb_logits_list[i], validate_args=False) for i in range(self.num_views)]
        reconstruction_loss = sum([distribution[i].log_prob(x[i]).sum(-1).mean() for i in range(self.num_views)]) # Maximum Likelihood Estimation (MLE)
        kld_loss = -0.5 * torch.sum(1 + logvar - mean.pow(2) - logvar.exp()) # Kullback-Leibler Divergence (KLD)
        return - reconstruction_loss + kld_loss

In [9]:
def predict_embedding_with_different_models(data, data_views, data_samples, data_features, data_categories):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_AE = AE(embed_dim=20, num_views=data_views, feature_dims=data_features, hidden_dims=[512, 512, 512]).to(device)
    model_DAE = DAE(embed_dim=20, num_views=data_views, feature_dims=data_features, hidden_dims=[512, 512, 512], noise_factor=0.1).to(device)
    model_VAE = VAE(embed_dim=20, num_views=data_views, feature_dims=data_features, hidden_dims=[512, 512, 512]).to(device)
    model_ZINBVAE = ZINBVAE(embed_dim=20, num_views=data_views, feature_dims=data_features, hidden_dims=[512, 512, 512], distribution='zinb').to(device)
    model_dict = {'AE': model_AE, 'DAE': model_DAE, 'VAE': model_VAE, 'ZINBVAE': model_ZINBVAE}
    embeddings_dict = {}
    for model_name, model in model_dict.items():
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        for epoch in range(100):
            model.train()
            optimizer.zero_grad()
            loss = model.reconstruction_loss(data)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            embeddings = model.forward_embedding(data).cpu().numpy()
        embeddings_dict[model_name] = embeddings

    return embeddings_dict

In [10]:
random.seed(0); np.random.seed(0); torch.manual_seed(0); torch.cuda.manual_seed_all(0) # Set random seed for reproducibility
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset = MMDataset('./data/data_sc_multiomics/TEA/', concat_data=False)
data = [x.clone().to(device) for x in dataset.X]; label = dataset.Y.clone().numpy()
data_views = dataset.data_views; data_samples = dataset.data_samples; data_features = dataset.data_features; data_categories = dataset.categories
label_dict_TEA = dataset.get_label_to_name()
label_TEA = label
embeddings_dict = predict_embedding_with_different_models(data, data_views, data_samples, data_features, data_categories)

modality_rna shape: (25517, 50)
modality_protein shape: (25517, 47)
modality_atac shape: (25517, 30)


In [ ]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import seaborn as sns
import scanpy as sc
import pandas as pd
import anndata as ad

def plot_umap_by_label_scanpy(embeddings, label, label_dict, save_dir=None):
    os.makedirs(save_dir, exist_ok=True)
    sc.settings.set_figure_params(dpi=400, facecolor='white', fontsize=14, frameon=False)
    plt.rcParams['font.family'] = 'DejaVu Sans'
    plt.rcParams['font.size'] = 14
    adata = ad.AnnData(embeddings, obs=pd.DataFrame(index=[f'cell_{i}' for i in range(embeddings.shape[0])]), var=pd.DataFrame(index=[f'embedding_{i}' for i in range(embeddings.shape[1])]))
    adata.obs['cell_type'] = [label_dict.get(int(l)) for l in label]
    sc.pp.neighbors(adata, use_rep='X')
    sc.tl.umap(adata, n_components=2)
    plt.figure(figsize=(6, 4))
    sc.pl.umap(adata, color='cell_type', size=20, show=False, frameon=True, title='')
    plt.title('', fontsize=14, fontweight='bold')
    plt.savefig(os.path.join(save_dir, 'umap_by_label.png'), dpi=400, bbox_inches='tight')
    plt.show()

# def plot_tsne_by_label(embeddings, label, label_dict, save_dir=None):
#     os.makedirs(save_dir, exist_ok=True)
#     tsne = TSNE(n_components=2, random_state=32, perplexity=10, max_iter=1000)
#     embeddings_2d = tsne.fit_transform(embeddings)
#     label_names = [label_dict.get(int(l)) for l in label]
#     unique_labels = np.unique(label)
#     unique_label_names = [label_dict.get(int(l)) for l in unique_labels]
#     colors_palette = plt.cm.tab10(np.linspace(0, 1, len(unique_labels))) if len(unique_labels) <= 10 else plt.cm.tab20(np.linspace(0, 1, len(unique_labels))) if len(unique_labels) <= 20 else sns.color_palette("husl", len(unique_labels))
#     label_to_color = {unique_label_names[i]: colors_palette[i] for i in range(len(unique_labels))}
#     plt.figure(figsize=(9, 6))
#     scatter = sns.scatterplot(x=embeddings_2d[:, 0], y=embeddings_2d[:, 1], hue=label_names, palette=label_to_color, s=20, edgecolor='k')
#     plt.xlabel('t-SNE component 1', fontsize=14)
#     plt.ylabel('t-SNE component 2', fontsize=14)
#     plt.xticks([])
#     plt.yticks([])
#     handles, labels_legend = scatter.get_legend_handles_labels()
#     plt.legend(handles=handles, labels=labels_legend, loc='center left', title='', bbox_to_anchor=(1.0, 0.5), frameon=False, prop={'size': 14, 'weight': 'normal'}, title_fontproperties={'size': 14, 'weight': 'bold'})
#     plt.tight_layout()
#     plt.savefig(os.path.join(save_dir, 'tsne_by_label.png'), dpi=400)
#     plt.show()
    
# plot_tsne_by_label(embeddings_dict['AE'], label_TEA, label_dict_TEA, './figure/hdur_sample_level/TEA/AE/')
plot_umap_by_label_scanpy(embeddings_dict['AE'], label_TEA, label_dict_TEA, './figure/hdur_sample_level/TEA/AE/')
# plot_tsne_by_label(embeddings_dict['DAE'], label_TEA, label_dict_TEA, './figure/hdur_sample_level/TEA/DAE/')
plot_umap_by_label_scanpy(embeddings_dict['DAE'], label_TEA, label_dict_TEA, './figure/hdur_sample_level/TEA/DAE/')
# plot_tsne_by_label(embeddings_dict['VAE'], label_TEA, label_dict_TEA, './figure/hdur_sample_level/TEA/VAE/')
# plot_umap_by_label_scanpy(embeddings_dict['VAE'], label_TEA, label_dict_TEA, './figure/hdur_sample_level/TEA/VAE/')
# plot_tsne_by_label(embeddings_dict['ZINBVAE'], label_TEA, label_dict_TEA, './figure/hdur_sample_level/TEA/ZINBVAE/')
# plot_umap_by_label_scanpy(embeddings_dict['ZINBVAE'], label_TEA, label_dict_TEA, './figure/hdur_sample_level/TEA/ZINBVAE/')

In [12]:
from _utils import clustering_acc, purity_score
from sklearn.cluster import KMeans
from sklearn.metrics import normalized_mutual_info_score, rand_score, adjusted_rand_score, silhouette_score

label_pred = KMeans(n_clusters=12, n_init=20).fit_predict(embeddings_dict['AE']) # KMeans is not a deterministic method and the results are random for different initializations.
acc = clustering_acc(label_TEA, label_pred) * 100; 
nmi = normalized_mutual_info_score(label_TEA, label_pred) * 100
ri = rand_score(label_TEA, label_pred) * 100; 
ari = adjusted_rand_score(label_TEA, label_pred) * 100
asw = silhouette_score(embeddings_dict['AE'], label_pred) * 100
purity = purity_score(label_TEA, label_pred) * 100
print(f'Model AE:: acc: {acc:.2f}, nmi: {nmi:.2f}, ri: {ri:.2f}, ari: {ari:.2f}, asw: {asw:.2f}, purity: {purity:.2f}')

label_pred = KMeans(n_clusters=12, n_init=20).fit_predict(embeddings_dict['DAE']) # KMeans is not a deterministic method and the results are random for different initializations.
acc = clustering_acc(label_TEA, label_pred) * 100; 
nmi = normalized_mutual_info_score(label_TEA, label_pred) * 100
ri = rand_score(label_TEA, label_pred) * 100; 
ari = adjusted_rand_score(label_TEA, label_pred) * 100
asw = silhouette_score(embeddings_dict['DAE'], label_pred) * 100
purity = purity_score(label_TEA, label_pred) * 100
print(f'Model DAE:: acc: {acc:.2f}, nmi: {nmi:.2f}, ri: {ri:.2f}, ari: {ari:.2f}, asw: {asw:.2f}, purity: {purity:.2f}')

label_pred = KMeans(n_clusters=12, n_init=20).fit_predict(embeddings_dict['VAE']) # KMeans is not a deterministic method and the results are random for different initializations.
acc = clustering_acc(label_TEA, label_pred) * 100; 
nmi = normalized_mutual_info_score(label_TEA, label_pred) * 100
ri = rand_score(label_TEA, label_pred) * 100; 
ari = adjusted_rand_score(label_TEA, label_pred) * 100
asw = silhouette_score(embeddings_dict['VAE'], label_pred) * 100
purity = purity_score(label_TEA, label_pred) * 100
print(f'Model VAE:: acc: {acc:.2f}, nmi: {nmi:.2f}, ri: {ri:.2f}, ari: {ari:.2f}, asw: {asw:.2f}, purity: {purity:.2f}')

label_pred = KMeans(n_clusters=12, n_init=20).fit_predict(embeddings_dict['ZINBVAE']) # KMeans is not a deterministic method and the results are random for different initializations.
acc = clustering_acc(label_TEA, label_pred) * 100; 
nmi = normalized_mutual_info_score(label_TEA, label_pred) * 100
ri = rand_score(label_TEA, label_pred) * 100; 
ari = adjusted_rand_score(label_TEA, label_pred) * 100
asw = silhouette_score(embeddings_dict['ZINBVAE'], label_pred) * 100
purity = purity_score(label_TEA, label_pred) * 100
print(f'Model ZINBVAE:: acc: {acc:.2f}, nmi: {nmi:.2f}, ri: {ri:.2f}, ari: {ari:.2f}, asw: {asw:.2f}, purity: {purity:.2f}')


Model AE:: acc: 57.24, nmi: 64.36, ri: 87.29, ari: 45.09, asw: 27.30, purity: 78.16
Model DAE:: acc: 58.82, nmi: 63.74, ri: 85.15, ari: 45.34, asw: 26.57, purity: 76.60
Model VAE:: acc: 12.35, nmi: 1.87, ri: 78.62, ari: 0.97, asw: 3.91, purity: 25.45
Model ZINBVAE:: acc: 12.66, nmi: 1.50, ri: 78.57, ari: 0.77, asw: 3.81, purity: 25.12
